In [1]:
import os
import numpy as np
import pandas as pd
import mne
from scipy.signal import welch

In [2]:
def get_edf_files(data_folder):
    edf_files = sorted([
        file for file in os.listdir(data_folder)
        if file.endswith(".edf")
    ])
    return edf_files

In [3]:
data_folder ="../data/chb01"
edf_files = get_edf_files(data_folder)
print("Total EDF Files:", len(edf_files))
for file in edf_files:
    print(file)

Total EDF Files: 15
chb01_01.edf
chb01_03.edf
chb01_04.edf
chb01_09.edf
chb01_15.edf
chb01_18.edf
chb01_21.edf
chb01_26.edf
chb01_30.edf
chb01_38.edf
chb01_39.edf
chb01_40.edf
chb01_41.edf
chb01_42.edf
chb01_46.edf


In [4]:
#  Function to read edf file

def read_edf(file_path):
    raw = mne.io.read_raw_edf(
        file_path,
        preload=True,
        verbose=False
    )
    return raw 

In [5]:
first_file = edf_files[0]
file_path = os.path.join(data_folder, first_file)
raw = read_edf(file_path)
print("Files:", first_file)
print("Sampling Frequency:", raw.info["sfreq"])
print("Channels:", len(raw.ch_names))
print("Samples:", raw.n_times)

C:\Users\Prajapati_Shivam\AppData\Local\Temp\ipykernel_8704\715924778.py:4: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(


Files: chb01_01.edf
Sampling Frequency: 256.0
Channels: 23
Samples: 921600


In [6]:
def preprocess_signal(raw):
    raw = raw.copy()
    raw.filter(
        l_freq=0.5,
        h_freq=40,
        verbose=False
    )
    return raw
    

In [7]:
processed_raw = preprocess_signal(raw)
print("Original Samples :",raw.n_times)
print("Processed Samples:",processed_raw.n_times)
print("Sampling Frequency:",processed_raw.info["sfreq"])

Original Samples : 921600
Processed Samples: 921600
Sampling Frequency: 256.0


In [8]:
def create_windows(raw, window_size=4):
 data = raw.get_data()
    sfreq = raw.info["sfreq"]
    samples_per_window = int(window_size * sfreq)
    windows = []
    total_samples = data.shape[1]
    for start in range(0, total_samples, samples_per_window):
        end = start + samples_per_window
        if end <= total_samples:
            window = data[:, start:end]
            windows.append(window)
    return np.array(windows)

In [9]:
windows = create_windows(processed_raw)
print("Total Windows:", len(windows))
print("Shape:", windows[0].shape)

Total Windows: 900
Shape: (23, 1024)


In [10]:
def label_windows(total_windows,
                  window_size,
                  seizure_start,
                  seizure_end):

    labels = []
    for i in range(total_windows):
        window_start = i * window_size
        window_end = window_start + window_size
        if window_start < seizure_end and window_end > seizure_start:
            labels.append(1)
        else:
            labels.append(0)
    return labels

In [11]:
def label_windows(total_windows,
                  window_size,
                  seizure_start,
                  seizure_end):
    labels = []
    for i in range(total_windows):
        window_start = i * window_size
        window_end = window_start + window_size
        if (window_start <= seizure_end) and (window_end >= seizure_start):
            labels.append(1)
        else:
            labels.append(0)
    return labels

In [12]:
window_size = 4
labels = label_windows(
    total_windows=len(windows),
    window_size=window_size,
    seizure_start=2996,
    seizure_end=3036
)
print("Total Labels:", len(labels))
print("Normal Windows:", labels.count(0))
print("Seizure Windows:", labels.count(1))

Total Labels: 900
Normal Windows: 888
Seizure Windows: 12


In [13]:
print(len(windows))

900


In [16]:
import numpy as np
# Seizure interval from CHB-MIT annotation
seizure_start = 2996
seizure_end = 3036
window_size = 4
labels = []
for i in range(len(windows)):
    window_start = i * window_size
    window_end = window_start + window_size
    # Label 1 only if the entire window is inside seizure interval
    if window_start >= seizure_start and window_end <= seizure_end:
        labels.append(1)
    else:
        labels.append(0)
# Convert to NumPy array
labels = np.array(labels)
# Check results
print("Total Windows:", len(labels))
print("Normal Windows:", np.sum(labels == 0))
print("Seizure Windows:", np.sum(labels == 1))
# Get seizure indices
seizure_indices = np.where(labels == 1)[0]
print("\nSeizure Window Indices:")
print(seizure_indices)
print("\nSeizure Window Time Ranges:")
for i in seizure_indices:
    start = i * window_size
    end = start + window_size
    print(f"Window {i}: {start}-{end} seconds")

Total Windows: 900
Normal Windows: 890
Seizure Windows: 10

Seizure Window Indices:
[749 750 751 752 753 754 755 756 757 758]

Seizure Window Time Ranges:
Window 749: 2996-3000 seconds
Window 750: 3000-3004 seconds
Window 751: 3004-3008 seconds
Window 752: 3008-3012 seconds
Window 753: 3012-3016 seconds
Window 754: 3016-3020 seconds
Window 755: 3020-3024 seconds
Window 756: 3024-3028 seconds
Window 757: 3028-3032 seconds
Window 758: 3032-3036 seconds
